# DAQ Driver Test Notebook

Run each cell from top to bottom. This tests the important `Daq` commands on the connected hardware.

---
## 0. Setup & Connection

In [ ]:
import numpy as np
from piec.drivers.autodetect import autodetect

In [ ]:
# Option A: detect the first connected MCC DAQ
daq = autodetect(verbose=True)

# Option B: connect directly by MCC board number
# from piec.drivers.daq.usb1208hs import USB1208HS
# daq = USB1208HS(0, verbose=True)

assert daq is not None, "No DAQ detected"

---
## 1. Instrument Tests

### 1.1 Identification (`idn`)

In [ ]:
print("IDN:", daq.idn())

### 1.2 Basic Instrument Commands

In [ ]:
daq.reset()
daq.clear()
print("Error:", daq.error())
print("Self-test:", daq.self_test())
daq.wait()
print("Operation complete:", daq.operation_complete())
daq.initialize()

### 1.3 Capabilities

In [ ]:
print("AI channels:", daq.ai_channel)
print("AI ranges:", daq.ai_range)
print("AI modes:", daq.ai_mode)
print("AI sample rate:", daq.ai_sample_rate)
print("AO channels:", daq.ao_channel)
print("AO ranges:", daq.ao_range)
print("AO sample rate:", daq.ao_sample_rate)
print("DIO channels:", daq.dio_channel)

---
## 2. Analog Input Tests

Connect a known voltage to AI0. The following uses the first supported input mode and range.

### 2.1 Individual AI Settings

In [ ]:
AI_CHANNEL = daq.ai_channel[0] if len(daq.ai_channel) != 0 else None

if AI_CHANNEL is not None:
    AI_RANGE = daq.ai_range[0]
    AI_RATE = 1000
    daq.set_AI_channel(AI_CHANNEL)
    daq.set_AI_range(AI_CHANNEL, AI_RANGE)
    daq.set_AI_sample_rate(AI_CHANNEL, AI_RATE)
    print("AI settings applied")

### 2.2 Combined AI Configuration (`configure_AI_channel`)

In [ ]:
if AI_CHANNEL is not None:
    daq.configure_AI_channel(
        channel=AI_CHANNEL,
        range=AI_RANGE,
        sample_rate=AI_RATE,
    )
    print("AI channel configured")

### 2.3 Single Read (`read_AI`, `quick_read`, `read_data`)

In [ ]:
if AI_CHANNEL is not None:
    print("read_AI:", daq.read_AI(AI_CHANNEL), "V")
    print("quick_read:", daq.quick_read(), "V")
    print("read_data:", daq.read_data(AI_CHANNEL), "V")

### 2.4 Analog Input Scan (`read_AI_scan`)

In [ ]:
if AI_CHANNEL is not None:
    scan = np.asarray(daq.read_AI_scan(AI_CHANNEL, points=1000, rate=AI_RATE))
    print("Points:", len(scan))
    print("Mean voltage:", scan.mean(), "V")

---
## 3. Analog Output Tests

Connect AO0 to a high-impedance oscilloscope input. Run the zero-output cell after each test. The plain USB-1208HS has no analog outputs and will skip these cells.

### 3.1 AO Configuration

In [ ]:
AO_CHANNEL = daq.ao_channel[0] if len(daq.ao_channel) != 0 else None

if AO_CHANNEL is not None:
    AO_RANGE = daq.ao_range[0]
    AO_RATE = 1000
    daq.set_AO_channel(AO_CHANNEL)
    daq.set_AO_range(AO_CHANNEL, AO_RANGE)
    daq.set_AO_sample_rate(AO_CHANNEL, AO_RATE)
    daq.configure_AO_channel(AO_CHANNEL, range=AO_RANGE, sample_rate=AO_RATE)
    print("AO channel configured")
else:
    print("Skipped: this DAQ has no analog outputs")

### 3.2 Static Output (`write_AO`)

In [ ]:
if AO_CHANNEL is not None:
    daq.write_AO(AO_CHANNEL, 0.1)
    print("AO output set to 0.1 V; verify on the oscilloscope")

In [ ]:
if AO_CHANNEL is not None:
    daq.write_AO(AO_CHANNEL, 0.0)
    print("AO output returned to 0 V")

### 3.3 Waveform Output (`write_waveform_scan`)

In [ ]:
if AO_CHANNEL is not None:
    waveform = 0.1 * np.sin(np.linspace(0, 2 * np.pi, 100, endpoint=False))
    daq.write_waveform_scan(AO_CHANNEL, waveform, sample_rate=1000)
    print("Waveform started; verify it on the oscilloscope")

In [ ]:
if AO_CHANNEL is not None:
    daq.stop_output()
    daq.write_AO(AO_CHANNEL, 0.0)
    print("Waveform stopped; AO returned to 0 V")

---
## 4. Digital I/O Test

Optional: connect the first DIO line to the second DIO line for a loopback test.

In [ ]:
if len(daq.dio_channel) >= 2:
    DO_CHANNEL, DI_CHANNEL = daq.dio_channel[:2]
    daq.set_DIO_channel(DO_CHANNEL)
    daq.set_DIO_mode(DO_CHANNEL, "O")
    daq.configure_DO_channel(DO_CHANNEL)
    daq.configure_DI_channel(DI_CHANNEL)
    daq.write_DO(DO_CHANNEL, 1)
    print("Digital input:", daq.read_DI(DI_CHANNEL))

In [ ]:
if len(daq.dio_channel) >= 2:
    daq.write_DO(DO_CHANNEL, 0)
    print("Digital output returned low")

---
## 5. Optional Features

### 5.1 Analog Input Mode (`set_input_mode`)

In [ ]:
if daq.ai_mode:
    daq.set_input_mode(daq.ai_mode[0])
    print("Input mode:", daq.ai_mode[0])
else:
    print("Skipped: input mode is not configurable")

---
## 6. Cleanup

In [ ]:
if len(daq.ao_channel) != 0:
    daq.stop_output()
    for channel in daq.ao_channel:
        daq.write_AO(channel, 0.0)
daq.close()
print("DAQ outputs set safe and connection closed")